# Judge-vs-Pathologist Agreement (Phase 1: Benchmarks 1-4)

Validates InternVL3.5-38B and Qwen3-VL-32B-Thinking as judges by comparing their
scores against the human pathologist ratings already collected, for every row the
pathologists rated (all 5 evaluators' assigned parts pooled, ~231 PathOPEN core
rows / 174 augmentation rows / 662 filtered-PathVQA rows).

**Important**: each row was rated by exactly one of the 5 pathologists (the parts
are disjoint, verified separately - zero row overlap between `evaluator1`..`evaluator5`
CSVs). So this is not classic multi-rater IRR; it is "judge score vs. whichever single
pathologist rated this row", pooled across the full dataset to get one agreement
statistic per (benchmark x criterion x dataset).

For each (benchmark, criterion, dataset) combination this notebook computes:
- **Weighted Cohen's kappa** (quadratic weights, appropriate for ordinal -1..2 data)
  between judge score and human score.
- **Mann-Whitney U test + rank-biserial effect size**, comparing the score
  *distribution* on PathOPEN vs. filtered-PathVQA (Pillar 1a's stated method),
  computed separately for human ratings and for each judge.

Rows rated `-1` by the human ("unable to comprehend / evaluate") are excluded from
kappa, consistent with the paper's own methodology for tiering (Pillar 2) and
generally for any question/answer the rater found uninterpretable.

**Prerequisite**: run `judge_runner_pathopen.ipynb` and `judge_runner_pathvqa.ipynb`
first so `judge_output/evaluator_internvl/` and `judge_output/evaluator_qwenvl/`
exist.


In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.metrics import cohen_kappa_score


In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
HUMAN_INPUT_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "scoring_analysis", "input"
)
JUDGE_OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")
JUDGE_KEYS = ["internvl", "qwenvl"]

REPO_ROOT, HUMAN_INPUT_DIR, JUDGE_OUTPUT_DIR


## Load and pool human ratings

Pools `evaluator1`..`evaluator5`'s files per dataset into one table (`__evaluator__`
column retained for traceability), dropping the benchmark-label header row that each
raw CSV carries at index 0 (same convention as the existing
`*_scoring_analysis_individual.ipynb` notebooks).

For PathOPEN, `Image_ID` is unique across the pooled set (verified separately - the
5 evaluators' files are disjoint row slices with no overlap), so it is a safe join
key on its own.

For **filtered PathVQA, `image_id` is NOT unique** (the same image hosts several
different questions - 224 duplicate `Image_ID` values across the pooled human
ratings). Each `evaluatorN/pathvqa_eval_data.csv` is row-for-row aligned with
`subsets_processing_output/data/pathvqa_partN.csv` (verified: identical question
text at every row position, per evaluator). So a global `row_uid` is reconstructed
here by concatenating `evaluator1`..`evaluator5` in that exact numeric order,
matching how `judge_runner_pathvqa.ipynb` builds its own `row_uid` by concatenating
`pathvqa_part1`..`part5` in sorted order.

In [ ]:
def _evaluator_dirs_in_order() -> list:
    """evaluator1..evaluator5 in strict numeric order (not lexicographic - avoids a
    latent evaluator10-before-evaluator2 bug if the evaluator count ever grows past 9)."""
    dirs = glob.glob(os.path.join(HUMAN_INPUT_DIR, "evaluator[0-9]*"))
    return sorted(dirs, key=lambda p: int(os.path.basename(p).replace("evaluator", "")))


def load_pooled_human(dataset_filename: str, add_row_uid: bool = False) -> pd.DataFrame:
    frames = []
    for evaluator_dir in _evaluator_dirs_in_order():
        path = os.path.join(evaluator_dir, dataset_filename)
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df = df.drop(index=0).reset_index(drop=True)  # drop benchmark-label header row
        df["__evaluator__"] = os.path.basename(evaluator_dir)
        frames.append(df)
    pooled = pd.concat(frames, ignore_index=True)
    if add_row_uid:
        pooled["row_uid"] = range(len(pooled))
    return pooled


human_pathopen = load_pooled_human("pathopen_eval_data.csv")
human_pathvqa = load_pooled_human("pathvqa_eval_data.csv", add_row_uid=True)
human_pathopen.shape, human_pathvqa.shape


In [ ]:
def load_judge(model_key: str, dataset_filename: str) -> pd.DataFrame:
    path = os.path.join(JUDGE_OUTPUT_DIR, f"evaluator_{model_key}", dataset_filename)
    return pd.read_csv(path)


judge_pathopen = {k: load_judge(k, "pathopen_eval_data.csv") for k in JUDGE_KEYS}
judge_pathvqa = {k: load_judge(k, "pathvqa_eval_data.csv") for k in JUDGE_KEYS}
{k: v.shape for k, v in judge_pathopen.items()}


## Column mapping: which (benchmark, criterion) each pair of columns represents

Mirrors the rename step in `pathopen_scoring_analysis_individual.ipynb` /
`pathvqa_scoring_analysis_individual.ipynb`, but keeps a `(dataset, human_column,
judge_column, benchmark, criterion, question_type)` mapping explicit so both the
kappa and Mann-Whitney computations can iterate it directly.

In [ ]:
# Each entry: (question_type, human_column, judge_column, benchmark, criterion)
PATHOPEN_COLUMN_MAP = []
for i in (1, 2):
    PATHOPEN_COLUMN_MAP.append(("OE_correct", f"Evaluation OE_Correct_Answer_{i}\n(Benchmark 1)",
                                 f"Evaluation OE_Correct_Answer_{i}\n(Benchmark 1)", 1, "Knowledge Interpretation/Deduction"))
    PATHOPEN_COLUMN_MAP.append(("OE_correct", f"Unnamed: {6 if i == 1 else 13}",
                                 f"OE_Correct_Answer_{i}_VisGround", 1, "Visual Grounding"))
    PATHOPEN_COLUMN_MAP.append(("OE_wrong", f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)",
                                 f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)", 2, "Error Proximity and Deductive Plausibility"))
    PATHOPEN_COLUMN_MAP.append(("OE_wrong", f"Unnamed: {9 if i == 1 else 16}",
                                 f"OE_Wrong_Answer_{i}_VisGroundErr", 2, "Visual Grounding Error"))

PATHOPEN_COLUMN_MAP.append(("MCQ_correct", "Evaluation MCQ_OE_Correct_Answer\n(Benchmark 1)",
                             "Evaluation MCQ_OE_Correct_Answer\n(Benchmark 1)", 1, "Knowledge Interpretation/Deduction"))
PATHOPEN_COLUMN_MAP.append(("MCQ_correct", "Unnamed: 20", "MCQ_OE_Correct_Answer_VisGround", 1, "Visual Grounding"))

for i in (1, 2, 3, 4):
    unnamed_idx = {1: 23, 2: 26, 3: 29, 4: 32}[i]
    PATHOPEN_COLUMN_MAP.append(("MCQ_wrong", f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)",
                                 f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)", 2, "Error Proximity and Deductive Plausibility"))
    PATHOPEN_COLUMN_MAP.append(("MCQ_wrong", f"Unnamed: {unnamed_idx}",
                                 f"MCQ_OE_Wrong_Answer_{i}_VisGroundErr", 2, "Visual Grounding Error"))

PATHOPEN_COLUMN_MAP.append(("CE_correct", "Evaluation CE_Correct_Answer\n(Benchmark 3)",
                             "Evaluation CE_Correct_Answer\n(Benchmark 3)", 3, "Visual Grounding/Reasoning"))

PATHVQA_COLUMN_MAP = [
    ("OE_correct", "Evaluation OE_Correct_Answer\n(Benchmark 1)", "Evaluation OE_Correct_Answer\n(Benchmark 1)", 1, "Knowledge Interpretation/Deduction"),
    ("OE_correct", "Unnamed: 6", "OE_Correct_Answer_VisGround", 1, "Visual Grounding"),
    ("CE_correct", "Evaluation CE_Correct_Answer\n(Benchmark 3)", "Evaluation CE_Correct_Answer\n(Benchmark 3)", 3, "Visual Grounding/Reasoning"),
]

len(PATHOPEN_COLUMN_MAP), len(PATHVQA_COLUMN_MAP)


## Join judge scores to human scores

PathOPEN joins on `Image_ID` (unique across the pooled human dataset - verified
separately, no duplicates). Filtered PathVQA joins on the reconstructed `row_uid`
instead, since `Image_ID` repeats there (see note above).

Human and judge column names frequently collide (e.g. both call a column
`"Evaluation OE_Correct_Answer_1\n(Benchmark 1)"`), so after `merge(...,
suffixes=("_human", "_judge"))` pandas silently renames *both* copies of any
overlapping column - looking up the bare, unsuffixed name would raise a KeyError.
`_resolve_column` below always checks for the suffixed name first and only falls
back to the bare name when no collision occurred (e.g. judge-only columns like
`OE_Correct_Answer_1_VisGround`, which have no human counterpart with that exact
name and thus never sprout a suffix).

In [ ]:
VALID_SCORES = {-1, 0, 1, 2}


def _resolve_column(merged: pd.DataFrame, col: str, side: str) -> pd.Series:
    suffixed = f"{col}_{side}"
    if suffixed in merged.columns:
        return pd.to_numeric(merged[suffixed], errors="coerce")
    return pd.to_numeric(merged[col], errors="coerce")


def compute_agreement(human_df: pd.DataFrame, judge_df: pd.DataFrame, column_map: list, dataset_name: str, join_key: str) -> pd.DataFrame:
    merged = human_df.merge(judge_df, on=join_key, suffixes=("_human", "_judge"))
    rows = []
    for question_type, human_col, judge_col, benchmark, criterion in column_map:
        if human_col not in human_df.columns or judge_col not in judge_df.columns:
            continue
        human_scores = _resolve_column(merged, human_col, "human")
        judge_scores = _resolve_column(merged, judge_col, "judge")

        valid_mask = human_scores.isin(VALID_SCORES) & judge_scores.isin(VALID_SCORES) & (human_scores != -1)
        n = int(valid_mask.sum())
        if n < 2:
            kappa = np.nan
        else:
            kappa = cohen_kappa_score(
                human_scores[valid_mask].astype(int),
                judge_scores[valid_mask].astype(int),
                weights="quadratic",
            )
        rows.append({
            "dataset": dataset_name,
            "question_type": question_type,
            "benchmark": benchmark,
            "criterion": criterion,
            "n": n,
            "weighted_kappa": kappa,
        })
    return pd.DataFrame(rows)


In [ ]:
agreement_tables = {}
for model_key in JUDGE_KEYS:
    pathopen_agreement = compute_agreement(human_pathopen, judge_pathopen[model_key], PATHOPEN_COLUMN_MAP, "PathOPEN", join_key="Image_ID")
    pathvqa_agreement = compute_agreement(human_pathvqa, judge_pathvqa[model_key], PATHVQA_COLUMN_MAP, "Filtered_PathVQA", join_key="row_uid")
    combined = pd.concat([pathopen_agreement, pathvqa_agreement], ignore_index=True)
    combined["judge"] = model_key
    agreement_tables[model_key] = combined

all_agreement = pd.concat(agreement_tables.values(), ignore_index=True)
all_agreement.sort_values(["dataset", "benchmark", "criterion", "judge"])


In [ ]:
os.makedirs(os.path.join(os.getcwd(), "agreement_output"), exist_ok=True)
all_agreement.to_csv(os.path.join(os.getcwd(), "agreement_output", "judge_pathologist_weighted_kappa.csv"), index=False)


## Distribution comparison: PathOPEN vs. Filtered-PathVQA (Mann-Whitney U)

Mirrors Pillar 1a's stated method exactly: compare score distributions between
PathOPEN and filtered-PathVQA on the criteria both datasets share (Benchmark 1:
Knowledge Interpretation/Deduction + Visual Grounding on OE correct answers;
Benchmark 3: Visual Grounding/Reasoning on CE correct answers), computed separately
for the human ratings and for each judge, with rank-biserial correlation as the
effect size.

In [ ]:
def rank_biserial_from_u(u_stat: float, n1: int, n2: int) -> float:
    """Rank-biserial correlation effect size derived from the Mann-Whitney U statistic."""
    return 1 - (2 * u_stat) / (n1 * n2)


def compare_pathopen_vs_pathvqa(pathopen_scores: pd.Series, pathvqa_scores: pd.Series) -> dict:
    a = pathopen_scores.dropna()
    a = a[a != -1]
    b = pathvqa_scores.dropna()
    b = b[b != -1]
    if len(a) < 2 or len(b) < 2:
        return {"n_pathopen": len(a), "n_pathvqa": len(b), "u_stat": np.nan, "p_value": np.nan, "rank_biserial": np.nan}
    u_stat, p_value = mannwhitneyu(a, b, alternative="two-sided")
    effect = rank_biserial_from_u(u_stat, len(a), len(b))
    return {"n_pathopen": len(a), "n_pathvqa": len(b), "u_stat": u_stat, "p_value": p_value, "rank_biserial": effect}


Human and judge CSVs use **different column names** for the same criterion
(the human CSV inherits pandas's `Unnamed: N` name for the paired Visual Grounding
score; the judge CSV names it explicitly, e.g. `OE_Correct_Answer_1_VisGround`), so
each (criterion, source) combination needs its own column name rather than one
shared name reused across human/judge frames.

In [ ]:
# label -> {source: (pathopen_col, pathvqa_col)}
shared_comparisons = {
    "OE_correct_KnowledgeInterpretation": {
        "human": ('Evaluation OE_Correct_Answer_1\n(Benchmark 1)', 'Evaluation OE_Correct_Answer\n(Benchmark 1)'),
        "judge": ('Evaluation OE_Correct_Answer_1\n(Benchmark 1)', 'Evaluation OE_Correct_Answer\n(Benchmark 1)'),
    },
    "OE_correct_VisualGrounding": {
        "human": ("Unnamed: 6", "Unnamed: 6"),
        "judge": ("OE_Correct_Answer_1_VisGround", "OE_Correct_Answer_VisGround"),
    },
    "CE_correct_VisualGrounding": {
        "human": ('Evaluation CE_Correct_Answer\n(Benchmark 3)', 'Evaluation CE_Correct_Answer\n(Benchmark 3)'),
        "judge": ('Evaluation CE_Correct_Answer\n(Benchmark 3)', 'Evaluation CE_Correct_Answer\n(Benchmark 3)'),
    },
}

mw_rows = []
for label, cols_by_source in shared_comparisons.items():
    human_pathopen_col, human_pathvqa_col = cols_by_source["human"]
    result = compare_pathopen_vs_pathvqa(
        pd.to_numeric(human_pathopen.get(human_pathopen_col), errors="coerce"),
        pd.to_numeric(human_pathvqa.get(human_pathvqa_col), errors="coerce"),
    )
    result.update({"criterion": label, "rater": "human"})
    mw_rows.append(result)

    judge_pathopen_col, judge_pathvqa_col = cols_by_source["judge"]
    for model_key in JUDGE_KEYS:
        result = compare_pathopen_vs_pathvqa(
            pd.to_numeric(judge_pathopen[model_key].get(judge_pathopen_col), errors="coerce"),
            pd.to_numeric(judge_pathvqa[model_key].get(judge_pathvqa_col), errors="coerce"),
        )
        result.update({"criterion": label, "rater": model_key})
        mw_rows.append(result)

mann_whitney_results = pd.DataFrame(mw_rows)
mann_whitney_results


In [ ]:
mann_whitney_results.to_csv(os.path.join(os.getcwd(), "agreement_output", "pathopen_vs_pathvqa_mannwhitney.csv"), index=False)


## Summary

`agreement_output/judge_pathologist_weighted_kappa.csv` reports, per (dataset,
benchmark, criterion, judge), the weighted Cohen's kappa between that judge and
whichever single pathologist rated each row, alongside n (after excluding human `-1`
ratings). Report these honestly even if only moderate, per the paper's own stated
reviewer concern.

`agreement_output/pathopen_vs_pathvqa_mannwhitney.csv` reports the Mann-Whitney U /
rank-biserial effect size for PathOPEN-vs-filtered-PathVQA score distributions, for
human ratings and each judge separately - this lets you see whether a judge
reproduces the *same qualitative conclusion* (PathOPEN scores higher) as the human
evaluators did, independent of raw kappa agreement on individual rows.
